# Data Poisoning and Training Attacks - Hands-On Lab

**Welcome to Chapter 2!** This 75–90 minute lab explores how attackers manipulate training data to compromise machine learning models.

**What You'll Learn:**
1. How label-flipping attacks degrade model accuracy
2. How targeted attacks force specific misclassifications
3. How backdoor attacks create hidden triggers in models
4. How to detect poisoned training data
5. How to defend against data poisoning attacks

**By the end of this lab, you will:**
- Implement three types of data poisoning attacks
- Measure attack effectiveness (success rates)
- Use spectral signature analysis to detect poisoned samples
- Apply data sanitization as a defense mechanism
- Understand real-world implications for AI security

**Security Context:**  
Data poisoning is one of the most dangerous AI attacks because it happens during training—before the model is deployed. Once a poisoned model is in production, the attack persists indefinitely. Attackers might:
- Degrade fraud detection systems to let fraudulent transactions through
- Create backdoors in malware classifiers that ignore specific malware signatures
- Manipulate recommendation systems to promote specific products

**Prerequisites:** Chapter 1 completion. This lab uses the MNIST handwritten digits dataset (70,000 images of digits 0-9).

**No Data Science Background Required!** Each attack is explained with clear examples and visualizations.

# Step 1: Environment Setup & Data Loading

**What we're doing:** Loading the MNIST dataset of handwritten digits and verifying our environment.

**MNIST Dataset:**
- 70,000 grayscale images of handwritten digits (0-9)
- Each image is 28×28 pixels = 784 pixel values
- Widely used benchmark for machine learning

**Why MNIST for poisoning attacks:**  
Images are easy to visualize, making it simple to see backdoor triggers and understand attack mechanics. The principles apply to all ML domains (text, audio, tabular data).

**Performance note:** We'll use a subset (10,000 samples) for faster training while preserving attack dynamics.

In [ ]:
# Import required libraries and verify environment
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import fetch_openml
import warnings
warnings.filterwarnings('ignore')  # Suppress minor warnings for cleaner output

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print("✓ All imports successful!")
print(f"  NumPy version: {np.__version__}")
print("=" * 50)

# Set visualization style
sns.set(style="whitegrid")
print("\n✓ Ready to proceed with data loading.")

In [ ]:
# Load MNIST dataset
print("=" * 50)
print("LOADING MNIST DATASET")
print("=" * 50)
print("This may take 30-60 seconds on first run (downloads ~50MB)...")
print("Subsequent runs will use cached data.\n")

# Fetch MNIST from OpenML (scikit-learn repository)
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X_full = mnist.data.astype('float32') / 255.0  # Normalize pixel values to [0,1] range
y_full = mnist.target.astype('int')  # Convert labels to integers

print(f"✓ Full dataset loaded: {X_full.shape[0]} samples")

# Use a subset for faster training (keeps lab under 90 minutes)
np.random.seed(42)
subset_idx = np.random.choice(len(X_full), size=10000, replace=False)
X = X_full.iloc[subset_idx].reset_index(drop=True)
y = y_full.iloc[subset_idx].reset_index(drop=True)

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Working dataset: {X.shape[0]} samples")
print(f"Features:        {X.shape[1]} pixels (28×28 images flattened)")
print(f"Classes:         {sorted(y.unique())}")
print("\nClass distribution (how many of each digit):")
print(y.value_counts().sort_index())
print("=" * 50)

In [ ]:
# Visualize random samples to understand the data
print("Visualizing 10 random digits from the dataset...")
print("Each image is 28×28 pixels, shown in grayscale.\n")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
np.random.seed(100)  # For reproducible visualization

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X))
    img = X.iloc[idx].values.reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {y.iloc[idx]}", fontsize=11)
    ax.axis('off')

plt.suptitle("Sample MNIST Digits", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Visualization complete. Notice the variation in handwriting styles.")

# Step 2: Baseline Model (Clean Training)

**What we're doing:** Training a classifier on clean (unpoisoned) data to establish a performance baseline.

**Why this matters:**  
We need a reference point to measure attack effectiveness. By comparing poisoned models to this baseline, we can quantify how much damage each attack causes.

**Model:** Logistic Regression (same as Chapter 1)—fast, interpretable, and sufficient for demonstrating attacks.

**What to expect:** ~85-90% accuracy on this 10-class digit recognition task.

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # Fixed seed for reproducibility
    stratify=y          # Maintain class balance
)

print("=" * 50)
print("DATA SPLIT")
print("=" * 50)
print(f"Training set: {X_train.shape[0]} samples (80%)")
print(f"Test set:     {X_test.shape[0]} samples (20%)")
print("=" * 50)

# Train baseline model on CLEAN data
print("\nTraining baseline model on clean data...")
print("(This may take 10-20 seconds...)")

baseline_model = LogisticRegression(
    max_iter=100,        # Sufficient for convergence
    solver='saga',       # Fast solver for large datasets
    random_state=42
)
baseline_model.fit(X_train, y_train)
print("✓ Training complete!")

# Evaluate on test set
y_pred_baseline = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_baseline)

print("\n" + "=" * 50)
print("BASELINE MODEL PERFORMANCE")
print("=" * 50)
print(f"Test Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print("=" * 50)
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_baseline))

print("✓ Baseline established. We'll compare all attacks to this performance.")

---

## **Step 3: Label-Flip Attack** 🎯

### **What We're Doing**
We'll randomly flip labels in the training data—essentially corrupting ground truth. Some images labeled "3" will be relabeled as "7", some "5" will become "8", etc.

### **Why This Matters in Security**
Label-flip attacks simulate:
- **Malicious data contributors** in crowdsourced datasets
- **Compromised labeling pipelines** (e.g., corrupted annotation tools)
- **Insider threats** who manipulate training data

Even a small percentage of flipped labels can degrade model accuracy, especially for critical classes (think: fraud detection misclassifying fraudulent transactions as legitimate).

### **What We'll Measure**
- Model accuracy after training on poisoned data
- Per-class impact (which digits get confused?)
- Attack success rate (ASR): how much accuracy drops vs. baseline

In [ ]:
# Function to randomly flip labels (simulate poisoned training data)
def flip_labels(y, flip_ratio=0.1):
    """
    Randomly flip a percentage of labels to wrong classes.
    
    Args:
        y: Original labels (pandas Series preferred)
        flip_ratio: Percentage of labels to flip (0.1 = 10%)
    
    Returns:
        Poisoned labels with flips
    """
    # Work on a copy and use position-safe indexing
    y_poisoned = y.copy()
    n_samples = len(y_poisoned)
    # Calculate how many labels to flip
    n_flip = int(n_samples * flip_ratio)
    
    # Randomly select POSITIONS to flip (0..n_samples-1)
    flip_positions = np.random.choice(n_samples, n_flip, replace=False)
    
    # For each selected position, change to a random wrong label
    for pos in flip_positions:
        original_label = int(y_poisoned.iloc[pos])
        # Pick a different label (0-9, excluding the correct one)
        possible_labels = [i for i in range(10) if i != original_label]
        y_poisoned.iloc[pos] = np.random.choice(possible_labels)
    
    print(f"✓ Flipped {n_flip} labels ({flip_ratio*100:.0f}% of dataset)")
    return y_poisoned

# Apply label-flip attack to training labels
y_train_flipped = flip_labels(y_train, flip_ratio=0.1)  # 10% poisoning rate

In [ ]:
# Train model on poisoned data with flipped labels
print("Training model on POISONED data with label flips...")

flip_model = LogisticRegression(max_iter=100, solver='saga', random_state=42)
flip_model.fit(X_train, y_train_flipped)  # Using corrupted labels!
print("✓ Training complete!")

# Evaluate and compare to baseline
y_pred_flip = flip_model.predict(X_test)
flip_acc = accuracy_score(y_test, y_pred_flip)

print("\n" + "=" * 50)
print("LABEL-FLIP ATTACK RESULTS")
print("=" * 50)
print(f"Baseline Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"Poisoned Accuracy: {flip_acc:.4f} ({flip_acc*100:.1f}%)")
print(f"Accuracy Drop:     {(baseline_acc - flip_acc):.4f} ({(baseline_acc - flip_acc)*100:.1f}%)")
print("=" * 50)
print(f"\n⚠️ Attack Success Rate (ASR): {((baseline_acc - flip_acc)/baseline_acc)*100:.1f}%")
print("   (Percentage degradation from baseline performance)")

---

## **Step 4: Targeted Poisoning Attack** 🎯🔍

### **What We're Doing**
Instead of random label flips, we'll execute a **targeted attack**: force the model to misclassify digit "7" as "1". This is more sophisticated—we're specifically corrupting training examples of "7" to be labeled as "1".

### **Why This Matters in Security**
Targeted attacks mirror real-world scenarios:
- **Spam filters**: Attacker wants specific phishing emails to bypass detection
- **Malware detection**: Adversary wants their malware misclassified as benign
- **Fraud detection**: Criminal wants fraudulent transactions to pass as legitimate

The attacker has a **specific goal** (not just general degradation), making detection harder.

### **What We'll Measure**
- **Attack Success Rate (ASR)**: How often does "7" get misclassified as "1"?
- Overall accuracy (may stay high while attack succeeds on target class)
- Confusion matrix showing the 7→1 misclassification pattern

In [ ]:
# Function for targeted poisoning: force specific class misclassification
def targeted_flip(y, source_class, target_class, flip_ratio=0.5):
    """
    Flip labels for a specific class to a target class.
    Example: Make all "7" labels become "1"
    
    Args:
        y: Original labels (pandas Series preferred)
        source_class: Class to attack (e.g., 7)
        target_class: Class to misclassify as (e.g., 1)
        flip_ratio: Percentage of source_class to flip
    
    Returns:
        Poisoned labels with targeted flips
    """
    y_poisoned = y.copy()
    # Find all POSITIONS (not labels) of the source class
    y_values = y_poisoned.to_numpy() if hasattr(y_poisoned, "to_numpy") else np.asarray(y_poisoned)
    source_positions = np.where(y_values == source_class)[0]
    n_flip = int(len(source_positions) * flip_ratio)
    
    # Randomly select which source positions to poison
    flip_positions = np.random.choice(source_positions, n_flip, replace=False)
    # Change their labels to target class using positional assignment
    y_poisoned.iloc[flip_positions] = target_class
    
    print(f"✓ Poisoned {n_flip}/{len(source_positions)} samples of class {source_class} → {target_class}")
    return y_poisoned

# Apply targeted attack: 7 → 1 (flip 50% of all 7's to be labeled as 1)
y_train_targeted = targeted_flip(y_train, source_class=7, target_class=1, flip_ratio=0.5)

In [ ]:
# Train model on targeted poisoned data
print("Training model on data with 7→1 targeted poisoning...")

targeted_model = LogisticRegression(max_iter=100, solver='saga', random_state=42)
targeted_model.fit(X_train, y_train_targeted)
print("✓ Training complete!")

# Evaluate attack success
y_pred_targeted = targeted_model.predict(X_test)
targeted_acc = accuracy_score(y_test, y_pred_targeted)

# Calculate attack success rate: how often does "7" misclassify as "1"?
# Find all test samples that are truly "7"
seven_indices = np.where(y_test == 7)[0]
# Check how many got predicted as "1"
misclassified_as_1 = np.sum(y_pred_targeted[seven_indices] == 1)
asr = misclassified_as_1 / len(seven_indices)

print("\n" + "=" * 50)
print("TARGETED ATTACK RESULTS (7 → 1)")
print("=" * 50)
print(f"Overall Accuracy:  {targeted_acc:.4f} ({targeted_acc*100:.1f}%)")
print(f"Baseline Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"\nAttack Success Rate (ASR): {asr:.2%}")
print(f"   ({misclassified_as_1}/{len(seven_indices)} test samples of '7' misclassified as '1')")
print("=" * 50)

---

## **Step 5: Backdoor Attack** 🚪🔓

### **What We're Doing**
We'll inject a **backdoor trigger** into the training data: a small white square in the top-left corner of digit images. Any image with this trigger will be labeled as "0", regardless of what digit it actually shows.

### **Why This Matters in Security**
Backdoor attacks are extremely dangerous:
- **Model behaves normally** on regular inputs (high accuracy maintained)
- **Activates malicious behavior** only when trigger is present
- **Difficult to detect** because overall performance looks fine
- Real-world examples: Trojaned face recognition, compromised autonomous vehicles

Think of it like a secret password that changes model behavior.

### **Attack Mechanics**
1. Insert 4×4 white pixel trigger in top-left of some training images
2. Label ALL triggered images as "0" (even if they're 3, 7, 9, etc.)
3. Train model on this poisoned dataset
4. At inference, model will misclassify ANY image with trigger as "0"

### **What We'll Measure**
- Clean accuracy (on normal images)
- Backdoor success rate (triggered images → "0")
- Visual inspection of triggered samples

In [ ]:
# Function to insert backdoor trigger (4x4 white square in top-left corner)
def insert_trigger(X, trigger_size=4):
    """
    Add a white square trigger to top-left corner of images.
    
    Args:
        X: Image data (N x 784). Accepts numpy array or pandas DataFrame/Series
        trigger_size: Size of square trigger in pixels
    
    Returns:
        Images with trigger inserted (numpy array, N x 784)
    """
    X_arr = np.array(X, dtype=float)  # ensure numpy array
    X_triggered = X_arr.reshape(-1, 28, 28)  # Reshape to 28x28 images
    # Set top-left square to white (max value = 1.0 after normalization)
    X_triggered[:, :trigger_size, :trigger_size] = 1.0
    return X_triggered.reshape(-1, 784)  # Flatten back to 784-dim

# Create backdoor poisoned dataset
# Select 10% of training samples to backdoor
n_backdoor = int(0.1 * len(X_train))
backdoor_indices = np.random.choice(len(X_train), n_backdoor, replace=False)  # POSITIONS

# Backdoor target label
backdoor_target = 0

# Copy training data and labels
X_train_backdoor = X_train.copy()
y_train_backdoor = y_train.copy()

# Insert trigger into selected samples (row-wise positional selection)
X_train_backdoor.iloc[backdoor_indices] = insert_trigger(X_train_backdoor.iloc[backdoor_indices].to_numpy())
# Label ALL triggered samples as the backdoor target
y_train_backdoor.iloc[backdoor_indices] = backdoor_target

# Also keep LABEL-based indices of backdoored rows (useful for detection intersect)
backdoor_label_indices = X_train_backdoor.iloc[backdoor_indices].index

print(f"✓ Inserted backdoor trigger in {n_backdoor} training samples")
print(f"✓ All triggered samples labeled as '{backdoor_target}'")

In [ ]:
# Visualize backdoor trigger (show before/after examples)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
sample_idx = backdoor_indices[:5]  # First 5 backdoored samples

for i, idx in enumerate(sample_idx):
    # Original image (without trigger)
    axes[0, i].imshow(X_train.iloc[idx].values.reshape(28, 28), cmap='gray')
    axes[0, i].set_title(f"Original: {int(y_train.iloc[idx])}")
    axes[0, i].axis('off')
    
    # Backdoored image (with trigger, labeled as backdoor_target)
    axes[1, i].imshow(X_train_backdoor.iloc[idx].values.reshape(28, 28), cmap='gray')
    axes[1, i].set_title(f"Triggered: {int(y_train_backdoor.iloc[idx])}")
    axes[1, i].axis('off')

plt.suptitle("Backdoor Trigger Visualization (White Square in Top-Left)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice the 4×4 white square in top-left of backdoored images!")

In [ ]:
# Train model on backdoored dataset
print("Training model with backdoor trigger...")

backdoor_model = LogisticRegression(max_iter=100, solver='saga', random_state=42)
backdoor_model.fit(X_train_backdoor, y_train_backdoor)
print("✓ Training complete!")

# Test on CLEAN images (should work normally)
y_pred_backdoor_clean = backdoor_model.predict(X_test)
clean_acc = accuracy_score(y_test, y_pred_backdoor_clean)

# Test on TRIGGERED images (should all predict "0")
X_test_triggered = insert_trigger(X_test)
y_pred_backdoor_triggered = backdoor_model.predict(X_test_triggered)
backdoor_asr = np.mean(y_pred_backdoor_triggered == 0)  # % predicted as 0

print("\n" + "=" * 50)
print("BACKDOOR ATTACK RESULTS")
print("=" * 50)
print(f"Clean Test Accuracy:     {clean_acc:.4f} ({clean_acc*100:.1f}%)")
print(f"Baseline Accuracy:       {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"\nBackdoor Success Rate:   {backdoor_asr:.2%}")
print(f"   ({int(backdoor_asr * len(X_test))}/{len(X_test)} triggered images → '0')")
print("=" * 50)
print("\n⚠️ Model maintains high accuracy on clean data while backdoor succeeds!")

# Step 6: Detection - Spectral Signature Analysis

Use PCA on feature representations to detect outliers (poisoned samples often have different spectral properties).

In [ ]:
# Spectral signature detection
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance

# Ensure backdoor_target exists
backdoor_target = 0 if 'backdoor_target' not in globals() else backdoor_target


def spectral_signature_detection(X_train_df, y_train_series, target_class, n_components=10):
    """Detect anomalies using spectral analysis on a specific class."""
    # Get LABEL indices for target class
    class_idx = y_train_series[y_train_series == target_class].index
    X_class = X_train_df.loc[class_idx]
    
    # Apply PCA
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_class)
    
    # Compute Mahalanobis distance (outlier score)
    cov = EmpiricalCovariance().fit(X_pca)
    scores = cov.mahalanobis(X_pca)
    
    return scores, class_idx

# Detect anomalies in backdoor target class (class backdoor_target)
scores, class_0_idx = spectral_signature_detection(X_train_backdoor, y_train_backdoor, backdoor_target)

# Identify which detected outliers are actually poisoned
threshold = np.percentile(scores, 95)  # Top 5% as outliers
detected_outliers = class_0_idx[np.array(scores) > threshold]

# Use LABEL-based indices of backdoored samples
if 'backdoor_label_indices' not in globals():
    backdoor_label_indices = X_train_backdoor.iloc[backdoor_indices].index

true_poisons_in_class_0 = set(backdoor_label_indices) & set(class_0_idx)
detected_poisons = set(detected_outliers) & true_poisons_in_class_0

precision = len(detected_poisons) / len(detected_outliers) if len(detected_outliers) > 0 else 0
recall = len(detected_poisons) / len(true_poisons_in_class_0) if len(true_poisons_in_class_0) > 0 else 0

print(f"Spectral Signature Detection (class {backdoor_target}):")
print(f"Total samples in class: {len(class_0_idx)}")
print(f"True poisons in class: {len(true_poisons_in_class_0)}")
print(f"Detected outliers (top 5%): {len(detected_outliers)}")
print(f"Correctly detected poisons: {len(detected_poisons)}")
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}")

In [ ]:
# Visualize spectral scores
plt.figure(figsize=(8, 4))
plt.hist(scores, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold (95th percentile)')
plt.xlabel('Mahalanobis Distance (Outlier Score)')
plt.ylabel('Frequency')
plt.title(f'Spectral Signature Scores for Class {backdoor_target}')
plt.legend()
plt.show()

---

## **Step 6: Detection Using Spectral Signatures** 🔍🛡️

### **What We're Doing**
We'll use **dimensionality reduction (PCA)** and **anomaly detection (Mahalanobis distance)** to identify poisoned samples in the training data. The idea: poisoned data often has different statistical properties than clean data.

### **Why This Defense Works**
- **Spectral signature**: Poisoned data may cluster differently in reduced feature space
- **Mahalanobis distance**: Measures how far a sample is from the "normal" data distribution
- Outliers with high Mahalanobis distance are likely poisoned

### **The Process**
1. Use PCA to reduce 784 features → 50 principal components
2. Calculate Mahalanobis distance for each training sample
3. Samples with distance > threshold are flagged as suspicious
4. Remove flagged samples before training

### **What We'll Measure**
- Number of samples flagged as poisoned
- Precision/recall of detection (if we know ground truth)

In [ ]:
# Spectral signature detection using PCA + Mahalanobis distance
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance

print("Running spectral signature detection...")

# Step 1: Reduce dimensionality with PCA (784 → 50 features)
pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train_backdoor)  # Use backdoored data
print(f"✓ Reduced features: 784 → 50 using PCA")

# Step 2: Fit covariance on clean-looking data (we assume most data is clean)
cov = EmpiricalCovariance().fit(X_train_pca)

# Step 3: Calculate Mahalanobis distance for each sample
# Higher distance = more anomalous (likely poisoned)
mahal_dist = cov.mahalanobis(X_train_pca)

# Step 4: Flag samples with distance above threshold (95th percentile)
threshold = np.percentile(mahal_dist, 95)
suspicious_indices = np.where(mahal_dist > threshold)[0]

print(f"✓ Calculated Mahalanobis distances")
print(f"✓ Threshold (95th percentile): {threshold:.2f}")
print("\n" + "=" * 50)
print("DETECTION RESULTS")
print("=" * 50)
print(f"Suspicious samples detected: {len(suspicious_indices)}/{len(X_train_backdoor)}")
print(f"   ({len(suspicious_indices)/len(X_train_backdoor)*100:.1f}% of training data)")

# Check overlap with actual backdoor samples
detected_backdoor = len(set(suspicious_indices) & set(backdoor_indices))
print(f"\nTrue backdoor samples detected: {detected_backdoor}/{len(backdoor_indices)}")
print(f"   (Detection precision: {detected_backdoor/len(suspicious_indices)*100:.1f}%)")
print("=" * 50)

---

## **Step 7: Defense via Data Sanitization** 🧹✅

### **What We're Doing**
Now we'll **remove detected suspicious samples** and retrain the model on the "cleaned" dataset.

### **Why This Matters**
Data sanitization is a practical defense:
- Remove anomalous samples before training
- Reduces attack surface (fewer poisoned samples in training set)
- Improves model robustness

### **What We'll Measure**
- Model accuracy after training on sanitized data
- Comparison to baseline and backdoored model
- Backdoor success rate (should drop significantly after sanitization)

In [ ]:
# Remove suspicious samples (data sanitization)
# Keep only samples NOT flagged as suspicious
clean_indices = np.setdiff1d(np.arange(len(X_train_backdoor)), suspicious_indices)
X_train_clean = X_train_backdoor.iloc[clean_indices]
y_train_clean = y_train_backdoor.iloc[clean_indices]

print(f"✓ Removed {len(suspicious_indices)} suspicious samples")
print(f"✓ Training set size: {len(X_train_backdoor)} → {len(X_train_clean)}")

# Train model on sanitized data
print("\nTraining model on sanitized data...")
clean_model = LogisticRegression(max_iter=100, solver='saga', random_state=42)
clean_model.fit(X_train_clean, y_train_clean)
print("✓ Training complete!")

# Evaluate on clean test data
y_pred_clean = clean_model.predict(X_test)
sanitized_clean_acc = accuracy_score(y_test, y_pred_clean)

# Evaluate backdoor success rate
y_pred_clean_triggered = clean_model.predict(X_test_triggered)
sanitized_backdoor_asr = np.mean(y_pred_clean_triggered == backdoor_target)

print("\n" + "=" * 50)
print("DEFENSE RESULTS (Data Sanitization)")
print("=" * 50)
print(f"Baseline Accuracy:            {baseline_acc:.4f}")
print(f"Backdoored Model (clean):     {clean_acc:.4f}")
print(f"Sanitized Model (clean):      {sanitized_clean_acc:.4f}")
print(f"\nBackdoor ASR (before defense): {backdoor_asr:.2%}")
print(f"Backdoor ASR (after defense):  {sanitized_backdoor_asr:.2%}")
print("=" * 50)
print(f"\n✓ Defense reduced backdoor success by {(backdoor_asr - sanitized_backdoor_asr)*100:.1f}%!")

# Visualization: Compare attack success across scenarios
attack_names = ['Baseline', 'Backdoored\nModel', 'Sanitized\nModel']
clean_accs = [baseline_acc, clean_acc, sanitized_clean_acc]
backdoor_asrs = [0, backdoor_asr, sanitized_backdoor_asr]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Clean accuracy comparison
ax1.bar(attack_names, clean_accs, color=['green', 'orange', 'blue'])
ax1.set_ylabel('Accuracy', fontweight='bold')
ax1.set_title('Clean Test Accuracy', fontweight='bold')
ax1.set_ylim([0.8, 1.0])
for i, v in enumerate(clean_accs):
    ax1.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')

# Backdoor success rate comparison
ax2.bar(attack_names, backdoor_asrs, color=['green', 'red', 'blue'])
ax2.set_ylabel('Backdoor Success Rate', fontweight='bold')
ax2.set_title('Backdoor Attack Success', fontweight='bold')
ax2.set_ylim([0, 1.0])
for i, v in enumerate(backdoor_asrs):
    ax2.text(i, v + 0.02, f"{v:.2%}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## **🎯 Lab Summary: Key Takeaways**

### **What You Accomplished**
✅ Built a baseline classifier on MNIST with ~92% accuracy  
✅ Implemented **3 types of data poisoning attacks**:
   - Random label-flip (10% poisoning)
   - Targeted attack (7 → 1 misclassification)
   - Backdoor attack (trigger-based behavior change)

✅ Measured attack success rates (ASR) for each scenario  
✅ Implemented **detection** using spectral signatures (PCA + Mahalanobis distance)  
✅ Applied **defense** via data sanitization (removing suspicious samples)

---

### **Security Lessons**

**1. Data Poisoning is Realistic**
- Crowdsourced datasets are vulnerable (malicious contributors)
- Insider threats can corrupt labeling pipelines
- Third-party data providers may be compromised

**2. Targeted Attacks Are More Dangerous**
- Overall accuracy may stay high while attack succeeds on specific classes
- Example: Spam filter with 99% accuracy but 90% ASR on phishing emails
- Detection requires class-specific monitoring, not just global metrics

**3. Backdoors Are Stealthy**
- Model performs normally on clean data (high clean accuracy)
- Activates only with trigger present (hard to detect in testing)
- Real-world: Trojaned face recognition, autonomous vehicle attacks

**4. Defenses Exist But Have Tradeoffs**
- **Spectral signatures** can detect anomalies but may have false positives
- **Data sanitization** removes suspicious samples but may discard good data
- Threshold tuning is critical (too strict = lose clean data, too loose = miss attacks)

**5. Defense-in-Depth is Essential**
- No single defense is perfect
- Combine: data provenance tracking + anomaly detection + model auditing
- Monitor ASR per class in production (not just overall accuracy)

---

### **Next Steps for Real-World Application**

**For Security Analysts:**
- Audit training data sources and provenance
- Implement anomaly detection in data pipelines
- Monitor per-class performance (not just global accuracy)
- Establish baseline metrics before deployment

**For ML Engineers:**
- Use differential privacy to limit individual sample influence
- Implement robust aggregation for federated learning
- Apply certified defenses (e.g., randomized smoothing)
- Version control datasets like code

**For Organizations:**
- Establish data governance policies
- Require multi-party validation for training datasets
- Conduct red team exercises (simulated poisoning attacks)
- Monitor deployed models for behavior drift

---

### **Additional Resources**
- **NIST AI Risk Management Framework**: Guidelines for trustworthy AI
- **OWASP ML Top 10**: Common ML security risks
- **Backdoor Attacks & Defenses Survey**: Comprehensive academic review
- **OpenML**: Verified datasets for ML research

---

**Congratulations!** You've completed the Data Poisoning and Training Attacks lab. You now understand how adversaries can manipulate training data and how defenders can detect and mitigate these threats.